# Parent-cluster ML with total-balanced subsampled endmembers (`_EMtbs`)

This notebook revises the mask-v2 endmember experiment so that the unvegetated support does **not** dominate the response space.

## Response space

Nine classes only:

- vegetation Parent 1–8
- `NONVEGETATED`

The original endmember surface labels (`Sand`, `Playa`, `Dark Cliff`, etc.) are retained only as **sampling strata**. They are not separate classifier outputs in this experiment.

## Total-balanced endmember sampling

The nonvegetated class is constructed to have approximately the same total number of training observations as a typical vegetation parent:

1. filter endmember locations through mask v2;
2. require a static endmember location to have valid IA45 in all three vegetation-supported years:
   - 2016
   - 2018
   - 2025
3. calculate the median vegetation-parent sample size after mask-v2 filtering;
4. convert that total to a target number of static endmember locations;
5. sample those locations approximately evenly across the surviving endmember surface groups;
6. contribute the same three annual observations per selected location.

Therefore:

- `NONVEGETATED` has approximately one vegetation-parent-equivalent total sample size;
- 2016 / 2018 / 2025 are exactly balanced within the nonvegetated class;
- endmember surface groups are represented as evenly as their available point support allows;
- repeated years from a physical endmember location stay together in grouped CV.

## Robustness to arbitrary endmember selection

The notebook repeats the stratified endmember draw 20 times for grouped OOF evaluation. The final mapping model uses the fixed deterministic `RANDOM_STATE=42` draw rather than selecting the best-performing draw.

## Mask

`C:\NCA_DATA\templates\Masks\NCA_keepMask_v2.tif`

## Outputs

`C:\NCA_DATA\Analysis\PrelimML_Endmembers_MaskV2_EMtbs`

The final random forest writes a 9-class classification TIFF and a 3-band uncertainty TIFF for every year from 2016–2025.


In [12]:
from pathlib import Path
import re
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)

# =============================================================================
# PATHS
# =============================================================================

VEG_TRAINING_CSV = Path(
    r"A:\NCA_DATA\Validation\Supervised_State_Classification"
    r"\FieldVeg_K2_IA45_Training.csv"
)

HARMONIC_ROOT = Path(
    r"A:\NCA_DATA\S2_Harmonics"
)

KEEP_MASK = Path(
    r"C:\NCA_DATA\templates\Masks\NCA_keepMask_v2.tif"
)

OUTPUT_DIR = Path(
    r"C:\NCA_DATA\Analysis\PrelimML_Endmembers_MaskV2_EMtbs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ENDMEMBER_CANDIDATES = [
    Path(r"C:\NCA_DATA\Analysis\NCA_endmembers.csv"),
    Path(r"C:\NCA_DATA\NCA_endmembers.csv"),
    Path.cwd() / "NCA_endmembers.csv",
]

ENDMEMBER_CSV = next(
    (
        p for p in ENDMEMBER_CANDIDATES
        if p.exists()
    ),
    None,
)

if ENDMEMBER_CSV is None:
    raise FileNotFoundError(
        "Could not find NCA_endmembers.csv. "
        "Place it in C:\\NCA_DATA\\Analysis, C:\\NCA_DATA, "
        "or beside this notebook, or edit ENDMEMBER_CANDIDATES."
    )

VEG_YEARS = [
    2016,
    2018,
    2025,
]

YEARS_TO_MAP = list(
    range(
        2016,
        2026,
    )
)

EXPECTED_D = 45
VEG_PARENT_CODES = list(
    range(
        1,
        9,
    )
)

VEG_CLASSES = [
    f"VEG::{i}"
    for i in VEG_PARENT_CODES
]

NONVEG_CLASS = "NONVEGETATED"

ALL_CLASSES = (
    VEG_CLASSES
    + [
        NONVEG_CLASS,
    ]
)

CLASS_TO_CODE = {
    f"VEG::{i}": i
    for i in VEG_PARENT_CODES
}

CLASS_TO_CODE[
    NONVEG_CLASS
] = 9

CODE_TO_CLASS = {
    code: cls
    for cls, code in CLASS_TO_CODE.items()
}

RANDOM_STATE = 42
N_SPLITS = 5
N_EM_DRAWS = 20
COMPRESS = "DEFLATE"

MODEL_FOR_MAPPING = "random_forest"

# Litter remains explicitly outside this experiment.
ENDMEMBER_EXCLUDE_REGEX = r"litter"

print("Vegetation training:", VEG_TRAINING_CSV)
print("Endmembers:", ENDMEMBER_CSV)
print("Mask:", KEEP_MASK)
print("Output directory:", OUTPUT_DIR)
print("Vegetation-supported years:", VEG_YEARS)
print("Mapping years:", YEARS_TO_MAP)
print("Endmember resampling draws:", N_EM_DRAWS)


Vegetation training: A:\NCA_DATA\Validation\Supervised_State_Classification\FieldVeg_K2_IA45_Training.csv
Endmembers: C:\NCA_DATA\Analysis\NCA_endmembers.csv
Mask: C:\NCA_DATA\templates\Masks\NCA_keepMask_v2.tif
Output directory: C:\NCA_DATA\Analysis\PrelimML_Endmembers_MaskV2_EMtbs
Vegetation-supported years: [2016, 2018, 2025]
Mapping years: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Endmember resampling draws: 20


## 1. Resolve endmember surface groups

In [13]:
# =============================================================================
# ENDMEMBER TABLE
# =============================================================================

end_raw = pd.read_csv(
    ENDMEMBER_CSV,
    low_memory=False,
)

required_end = {
    "name",
    "EASTING",
    "NORTHING",
}

missing = required_end - set(
    end_raw.columns
)

if missing:
    raise KeyError(
        f"Endmember CSV missing columns: {sorted(missing)}"
    )

end_raw["name"] = (
    end_raw["name"]
    .astype(str)
    .str.strip()
)

end_raw["EASTING"] = pd.to_numeric(
    end_raw["EASTING"],
    errors="coerce",
)

end_raw["NORTHING"] = pd.to_numeric(
    end_raw["NORTHING"],
    errors="coerce",
)

if end_raw[
    [
        "EASTING",
        "NORTHING",
    ]
].isna().any().any():
    raise ValueError(
        "Endmember CSV contains missing/non-numeric coordinates."
    )

# Strip the replicate integer only:
# "Dark Cliff 17" -> "Dark Cliff"
end_raw[
    "endmember_group"
] = (
    end_raw["name"]
    .str.replace(
        r"\s+\d+$",
        "",
        regex=True,
    )
    .str.strip()
)

exclude_litter = (
    end_raw[
        "endmember_group"
    ]
    .str.contains(
        ENDMEMBER_EXCLUDE_REGEX,
        case=False,
        regex=True,
        na=False,
    )
)

if exclude_litter.any():
    print(
        "Excluding litter points:",
        int(
            exclude_litter.sum()
        )
    )

    end_raw = (
        end_raw.loc[
            ~exclude_litter
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

print(
    "Candidate endmember locations:",
    f"{len(end_raw):,}"
)

display(
    end_raw[
        "endmember_group"
    ]
    .value_counts()
    .rename_axis(
        "endmember_group"
    )
    .reset_index(
        name="n_candidate_locations"
    )
)


Candidate endmember locations: 221


,endmember_group,n_candidate_locations
0,Building,34
1,Dark Cliff,31
2,Sand,30
3,Bright Cliff,29
4,Solar,20
5,Water,20
6,Playa,19
7,Road,14
8,Pavement,12
9,BG/Parking Lot,7


## 2. Filter endmember locations through mask v2

In [14]:
# =============================================================================
# MASK SAMPLING
# =============================================================================

if not KEEP_MASK.exists():
    raise FileNotFoundError(
        f"Mask not found:\n{KEEP_MASK}"
    )


def mask_keep_for_points(
    table,
    x_col,
    y_col,
    mask_path,
):

    keep_flags = []
    mask_values = []

    with rasterio.open(
        mask_path
    ) as src:

        nodata = src.nodata

        for x, y in zip(
            table[x_col].to_numpy(),
            table[y_col].to_numpy(),
        ):

            inside = (
                src.bounds.left <= x <= src.bounds.right
                and src.bounds.bottom <= y <= src.bounds.top
            )

            if not inside:
                keep_flags.append(
                    False
                )
                mask_values.append(
                    np.nan
                )
                continue

            value = next(
                src.sample(
                    [
                        (
                            float(x),
                            float(y),
                        )
                    ],
                    indexes=1,
                    masked=False,
                )
            )[0]

            valid = np.isfinite(
                value
            )

            if nodata is not None:
                valid &= not np.isclose(
                    value,
                    nodata,
                )

            keep = (
                valid
                and value > 0
            )

            keep_flags.append(
                bool(
                    keep
                )
            )

            mask_values.append(
                float(value)
                if np.isfinite(
                    value
                )
                else np.nan
            )

    out = table.copy()

    out[
        "mask_value"
    ] = mask_values

    out[
        "keep_v2"
    ] = keep_flags

    return out


end_audit = mask_keep_for_points(
    end_raw,
    "EASTING",
    "NORTHING",
    KEEP_MASK,
)

end_audit.to_csv(
    OUTPUT_DIR
    / "NCA_endmembers_maskV2_audit_EMtbs.csv",
    index=False,
)

end_masked = (
    end_audit.loc[
        end_audit[
            "keep_v2"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

print(
    "Candidate:",
    len(
        end_audit
    )
)

print(
    "Retained by mask v2:",
    len(
        end_masked
    )
)

print(
    "Removed:",
    int(
        (
            ~end_audit[
                "keep_v2"
            ]
        ).sum()
    )
)

display(
    end_audit.groupby(
        "endmember_group",
        as_index=False,
    )
    .agg(
        n_candidate=(
            "name",
            "size",
        ),
        n_retained=(
            "keep_v2",
            "sum",
        ),
    )
    .sort_values(
        "n_retained",
        ascending=False,
    )
)


Candidate: 221
Retained by mask v2: 137
Removed: 84


,endmember_group,n_candidate,n_retained
5,Dark Cliff,31,31
9,Sand,30,30
2,Bright Cliff,29,26
7,Playa,19,18
6,Pavement,12,11
8,Road,14,7
1,BG/Parking Lot,7,7
3,Building,34,7
0,Asphalt,2,0
4,Cow,3,0


## 3. Load vegetation IA45 and apply the same mask

In [15]:
# =============================================================================
# VEGETATION TABLE
# =============================================================================

if not VEG_TRAINING_CSV.exists():
    raise FileNotFoundError(
        f"Vegetation training CSV not found:\n{VEG_TRAINING_CSV}"
    )

veg = pd.read_csv(
    VEG_TRAINING_CSV,
    low_memory=False,
).drop(
    columns=[
        "Unnamed: 0",
    ],
    errors="ignore",
)

veg["Year"] = pd.to_numeric(
    veg["Year"],
    errors="coerce",
)

veg[
    "parent_cluster"
] = pd.to_numeric(
    veg[
        "parent_cluster"
    ],
    errors="coerce",
)

veg = (
    veg.loc[
        veg["Year"].isin(
            VEG_YEARS
        )
        & veg[
            "parent_cluster"
        ].notna()
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

veg["Year"] = veg["Year"].astype(
    int
)

veg[
    "parent_cluster"
] = veg[
    "parent_cluster"
].astype(
    int
)

veg["PlotID"] = (
    veg["PlotID"]
    .astype(str)
    .str.strip()
)

if not all(
    c in veg.columns
    for c in [
        "Easting",
        "Northing",
    ]
):
    raise KeyError(
        "Vegetation training CSV must contain Easting and Northing "
        "for mask-v2 consistency."
    )

veg["Easting"] = pd.to_numeric(
    veg["Easting"],
    errors="coerce",
)

veg["Northing"] = pd.to_numeric(
    veg["Northing"],
    errors="coerce",
)

coord_ok = veg[
    [
        "Easting",
        "Northing",
    ]
].notna().all(
    axis=1
)

veg_coord = (
    veg.loc[
        coord_ok
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

veg_audit = mask_keep_for_points(
    veg_coord,
    "Easting",
    "Northing",
    KEEP_MASK,
)

veg_audit.to_csv(
    OUTPUT_DIR
    / "vegetation_training_maskV2_audit_EMtbs.csv",
    index=False,
)

veg = (
    veg_audit.loc[
        veg_audit[
            "keep_v2"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

print(
    "Vegetation rows retained:",
    len(
        veg
    )
)

veg_counts = (
    veg[
        "parent_cluster"
    ]
    .value_counts()
    .sort_index()
)

display(
    veg_counts.rename_axis(
        "parent_cluster"
    )
    .reset_index(
        name="n"
    )
)

TARGET_NONVEG_OBS = int(
    round(
        float(
            veg_counts.median()
        )
    )
)

TARGET_NONVEG_LOCATIONS = max(
    1,
    int(
        round(
            TARGET_NONVEG_OBS
            / len(
                VEG_YEARS
            )
        )
    ),
)

TARGET_NONVEG_OBS_ACTUAL = (
    TARGET_NONVEG_LOCATIONS
    * len(
        VEG_YEARS
    )
)

print(
    "\nMedian vegetation-parent n:",
    TARGET_NONVEG_OBS
)

print(
    "Target static endmember locations:",
    TARGET_NONVEG_LOCATIONS
)

print(
    "Target NONVEGETATED annual observations:",
    TARGET_NONVEG_OBS_ACTUAL
)


Vegetation rows retained: 467


,parent_cluster,n
0,1,80
1,2,59
2,3,73
3,4,82
4,5,74
5,6,50
6,7,30
7,8,19



Median vegetation-parent n: 66
Target static endmember locations: 22
Target NONVEGETATED annual observations: 66


## 4. Resolve IA45 and annual raster band matching

In [16]:
# =============================================================================
# IA45 + BAND MATCHING
# =============================================================================

intercept_cols = [
    c for c in veg.columns
    if re.search(
        r"^(constant_|intercept_)",
        str(c),
        flags=re.IGNORECASE,
    )
]

amplitude_cols = [
    c for c in veg.columns
    if re.search(
        r"^(amp|amplitude)",
        str(c),
        flags=re.IGNORECASE,
    )
]

IA_COLS = (
    intercept_cols
    + amplitude_cols
)

if len(
    IA_COLS
) != EXPECTED_D:
    raise ValueError(
        f"Expected {EXPECTED_D} IA predictors; "
        f"found {len(IA_COLS)}."
    )


def annual_k2_path(
    year,
):
    return (
        HARMONIC_ROOT
        / str(
            year
        )
        / (
            f"{year}"
            "_K2_InterceptAmpPhase_nobs_RMSE.tif"
        )
    )


def get_band_names(
    src,
):
    return [
        (
            str(
                desc
            ).strip()
            if desc is not None
            and str(
                desc
            ).strip()
            else f"band_{i}"
        )
        for i, desc in enumerate(
            src.descriptions,
            start=1,
        )
    ]


def normalize_name(
    x,
):
    return (
        str(
            x
        )
        .strip()
        .lower()
    )


def ia_band_indexes(
    src,
    ia_cols,
):

    names = get_band_names(
        src
    )

    lookup = {
        normalize_name(
            name
        ): i
        for i, name in enumerate(
            names,
            start=1,
        )
    }

    indexes = []
    missing = []

    for feature in ia_cols:

        key = normalize_name(
            feature
        )

        if key not in lookup:
            missing.append(
                feature
            )
        else:
            indexes.append(
                lookup[
                    key
                ]
            )

    if missing:
        raise KeyError(
            "Missing IA45 raster bands:\n"
            + "\n".join(
                missing
            )
        )

    if len(
        indexes
    ) != EXPECTED_D:
        raise RuntimeError(
            f"Expected {EXPECTED_D} bands; found {len(indexes)}."
        )

    return indexes


for year in YEARS_TO_MAP:

    path = annual_k2_path(
        year
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing annual raster:\n{path}"
        )

    with rasterio.open(
        path
    ) as src:

        idx = ia_band_indexes(
            src,
            IA_COLS,
        )

        print(
            year,
            "| IA45 bands:",
            len(
                idx
            ),
        )


2016 | IA45 bands: 45
2017 | IA45 bands: 45
2018 | IA45 bands: 45
2019 | IA45 bands: 45
2020 | IA45 bands: 45
2021 | IA45 bands: 45
2022 | IA45 bands: 45
2023 | IA45 bands: 45
2024 | IA45 bands: 45
2025 | IA45 bands: 45


## 5. Extract all mask-retained endmember locations for the three supported training years

Only locations with valid IA45 in **all three** of 2016, 2018, and 2025 are eligible for balanced subsampling. This guarantees exact temporal balance inside `NONVEGETATED`.


In [17]:
# =============================================================================
# ENDMEMBER IA45 — SUPPORTED YEARS ONLY
# =============================================================================

xy = list(
    zip(
        end_masked[
            "EASTING"
        ].astype(
            float
        ),
        end_masked[
            "NORTHING"
        ].astype(
            float
        ),
    )
)

endmember_rows = []

for year in VEG_YEARS:

    with rasterio.open(
        annual_k2_path(
            year
        )
    ) as src:

        indexes = ia_band_indexes(
            src,
            IA_COLS,
        )

        sampled = np.asarray(
            list(
                src.sample(
                    xy,
                    indexes=indexes,
                    masked=False,
                )
            ),
            dtype=np.float32,
        )

        valid = np.isfinite(
            sampled
        ).all(
            axis=1
        )

        if src.nodata is not None:
            valid &= ~np.any(
                np.isclose(
                    sampled,
                    src.nodata,
                ),
                axis=1,
            )

        print(
            year,
            "| valid:",
            int(
                valid.sum()
            ),
            "/",
            len(
                end_masked
            ),
        )

        for row_i in np.where(
            valid
        )[0]:

            row = {
                "name": str(
                    end_masked.loc[
                        row_i,
                        "name",
                    ]
                ),
                "endmember_group": str(
                    end_masked.loc[
                        row_i,
                        "endmember_group",
                    ]
                ),
                "Easting": float(
                    end_masked.loc[
                        row_i,
                        "EASTING",
                    ]
                ),
                "Northing": float(
                    end_masked.loc[
                        row_i,
                        "NORTHING",
                    ]
                ),
                "Year": int(
                    year
                ),
            }

            for j, feature in enumerate(
                IA_COLS
            ):
                row[
                    feature
                ] = float(
                    sampled[
                        row_i,
                        j,
                    ]
                )

            endmember_rows.append(
                row
            )


end_supported = pd.DataFrame(
    endmember_rows
)

coverage = (
    end_supported.groupby(
        [
            "name",
            "endmember_group",
        ],
        as_index=False,
    )
    .agg(
        n_years=(
            "Year",
            "nunique",
        ),
    )
)

complete_names = set(
    coverage.loc[
        coverage[
            "n_years"
        ].eq(
            len(
                VEG_YEARS
            )
        ),
        "name",
    ]
)

end_complete = (
    end_supported.loc[
        end_supported[
            "name"
        ].isin(
            complete_names
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

end_location_pool = (
    end_complete[
        [
            "name",
            "endmember_group",
            "Easting",
            "Northing",
        ]
    ]
    .drop_duplicates(
        subset=[
            "name",
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "\nStatic locations valid in all supported years:",
    len(
        end_location_pool
    )
)

display(
    end_location_pool[
        "endmember_group"
    ]
    .value_counts()
    .rename_axis(
        "endmember_group"
    )
    .reset_index(
        name="n_complete_locations"
    )
)

if len(
    end_location_pool
) < TARGET_NONVEG_LOCATIONS:
    raise RuntimeError(
        "Not enough complete static endmember locations to hit "
        "the requested NONVEGETATED target."
    )


2016 | valid: 88 / 137
2018 | valid: 92 / 137
2025 | valid: 83 / 137

Static locations valid in all supported years: 77


,endmember_group,n_complete_locations
0,Dark Cliff,31
1,Sand,29
2,Building,6
3,BG/Parking Lot,3
4,Bright Cliff,3
5,Road,3
6,Pavement,2


## 6. Even-strata static-location sampler

Sampling proceeds round-robin across the available endmember surface groups. This keeps the selected `NONVEGETATED` support as even across groups as available point support permits.


In [18]:
# =============================================================================
# BALANCED LOCATION SAMPLER
# =============================================================================

def balanced_endmember_locations(
    location_pool,
    target_n,
    seed,
):

    rng = np.random.default_rng(
        seed
    )

    classes = sorted(
        location_pool[
            "endmember_group"
        ]
        .unique()
        .tolist()
    )

    pools = {}

    for cls in classes:

        idx = (
            location_pool.index[
                location_pool[
                    "endmember_group"
                ].eq(
                    cls
                )
            ]
            .to_numpy()
        )

        pools[
            cls
        ] = rng.permutation(
            idx
        ).tolist()

    selected = []

    while len(
        selected
    ) < target_n:

        progressed = False

        for cls in classes:

            if len(
                selected
            ) >= target_n:
                break

            if pools[
                cls
            ]:

                selected.append(
                    pools[
                        cls
                    ].pop()
                )

                progressed = True

        if not progressed:
            break

    if len(
        selected
    ) < target_n:
        raise RuntimeError(
            f"Could sample only {len(selected)} of {target_n} "
            "requested locations."
        )

    return (
        location_pool.loc[
            selected
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


mapping_end_locations = balanced_endmember_locations(
    location_pool=end_location_pool,
    target_n=TARGET_NONVEG_LOCATIONS,
    seed=RANDOM_STATE,
)

print(
    "Mapping-draw endmember locations:",
    len(
        mapping_end_locations
    )
)

display(
    mapping_end_locations[
        "endmember_group"
    ]
    .value_counts()
    .rename_axis(
        "endmember_group"
    )
    .reset_index(
        name="n_selected_locations"
    )
)


Mapping-draw endmember locations: 22


,endmember_group,n_selected_locations
0,Building,4
1,Dark Cliff,4
2,BG/Parking Lot,3
3,Bright Cliff,3
4,Road,3
5,Sand,3
6,Pavement,2


## 7. Training-table constructor

In [19]:
# =============================================================================
# BUILD ONE BALANCED 9-CLASS TRAINING TABLE
# =============================================================================

veg_X = (
    veg[
        IA_COLS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)

if veg_X.isna().any(
    axis=1
).any():
    raise ValueError(
        "Retained vegetation table contains incomplete IA45."
    )


def make_training_table(
    selected_locations,
):

    selected_names = set(
        selected_locations[
            "name"
        ]
        .astype(
            str
        )
    )

    selected_end = (
        end_complete.loc[
            end_complete[
                "name"
            ].isin(
                selected_names
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    # Guarantee the static-location design contributed all 3 years.
    counts = (
        selected_end.groupby(
            "name"
        )[
            "Year"
        ]
        .nunique()
    )

    if not counts.eq(
        len(
            VEG_YEARS
        )
    ).all():
        raise RuntimeError(
            "A selected endmember location is missing a supported year."
        )

    veg_samples = pd.DataFrame({
        "sample_source": "vegetation",
        "sample_id": veg[
            "PlotID"
        ].astype(
            str
        ),
        "sample_group": (
            "VEG::"
            + veg[
                "PlotID"
            ].astype(
                str
            )
        ),
        "Year": veg[
            "Year"
        ].astype(
            int
        ),
        "class_name": (
            "VEG::"
            + veg[
                "parent_cluster"
            ].astype(
                int
            ).astype(
                str
            )
        ),
        "endmember_group": np.nan,
    })

    for feature in IA_COLS:
        veg_samples[
            feature
        ] = veg[
            feature
        ].astype(
            float
        ).to_numpy()

    end_samples = pd.DataFrame({
        "sample_source": "endmember",
        "sample_id": selected_end[
            "name"
        ].astype(
            str
        ),
        "sample_group": (
            "END::"
            + selected_end[
                "name"
            ].astype(
                str
            )
        ),
        "Year": selected_end[
            "Year"
        ].astype(
            int
        ),
        "class_name": NONVEG_CLASS,
        "endmember_group": selected_end[
            "endmember_group"
        ].astype(
            str
        ),
    })

    for feature in IA_COLS:
        end_samples[
            feature
        ] = selected_end[
            feature
        ].astype(
            float
        ).to_numpy()

    combined = pd.concat(
        [
            veg_samples,
            end_samples,
        ],
        ignore_index=True,
    )

    return combined


mapping_training = make_training_table(
    mapping_end_locations
)

print(
    "Mapping training rows:",
    len(
        mapping_training
    )
)

display(
    mapping_training[
        "class_name"
    ]
    .value_counts()
    .rename_axis(
        "class_name"
    )
    .reset_index(
        name="n"
    )
)

print(
    "\nNONVEGETATED by year:"
)

display(
    mapping_training.loc[
        mapping_training[
            "class_name"
        ].eq(
            NONVEG_CLASS
        )
    ][
        "Year"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "Year"
    )
    .reset_index(
        name="n"
    )
)


Mapping training rows: 533


,class_name,n
0,VEG::4,82
1,VEG::1,80
2,VEG::5,74
3,VEG::3,73
4,NONVEGETATED,66
5,VEG::2,59
6,VEG::6,50
7,VEG::7,30
8,VEG::8,19



NONVEGETATED by year:


,Year,n
0,2016,22
1,2018,22
2,2025,22


## 8. Candidate learners

In [20]:
# =============================================================================
# MODELS
# =============================================================================

MODELS = {
    "elastic_net_logistic": Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                solver="saga",
                penalty="elasticnet",
                l1_ratio=0.5,
                C=1.0,
                class_weight="balanced",
                max_iter=10000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "random_forest": RandomForestClassifier(
        n_estimators=800,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        bootstrap=True,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),

    "extra_trees": ExtraTreesClassifier(
        n_estimators=800,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced",
        bootstrap=False,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}


## 9. Evaluation helpers

In [21]:
# =============================================================================
# METRIC HELPERS
# =============================================================================

def score_predictions(
    y_true,
    y_pred,
):

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=ALL_CLASSES,
            average="macro",
            zero_division=0,
        ),
    }


def score_vegetation_only(
    y_true,
    y_pred,
):

    y_true = np.asarray(
        y_true,
        dtype=str,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=str,
    )

    mask = np.isin(
        y_true,
        VEG_CLASSES,
    )

    true_v = y_true[
        mask
    ]

    pred_v = y_pred[
        mask
    ]

    return {
        "veg_n": int(
            mask.sum()
        ),
        "veg_accuracy": accuracy_score(
            true_v,
            pred_v,
        ),
        "veg_balanced_accuracy": balanced_accuracy_score(
            true_v,
            pred_v,
        ),
        "veg_macro_f1": f1_score(
            true_v,
            pred_v,
            labels=VEG_CLASSES,
            average="macro",
            zero_division=0,
        ),
        "veg_predicted_nonveg_fraction": float(
            np.mean(
                pred_v
                == NONVEG_CLASS
            )
        ),
    }


## 10. Twenty balanced endmember draws: grouped OOF robustness

Each draw contains the same vegetation table and a new evenly stratified static-location sample for `NONVEGETATED`.

This answers whether vegetation performance is stable to the particular endmember locations selected.


In [ ]:
# =============================================================================
# REPEATED BALANCED-ENDMEMBER GROUPED CV
# =============================================================================

draw_rows = []

for draw in range(
    N_EM_DRAWS
):

    draw_seed = (
        RANDOM_STATE
        + draw
    )

    selected_locations = balanced_endmember_locations(
        location_pool=end_location_pool,
        target_n=TARGET_NONVEG_LOCATIONS,
        seed=draw_seed,
    )

    training = make_training_table(
        selected_locations
    )

    X_draw = training[
        IA_COLS
    ].to_numpy(
        dtype=np.float32
    )

    y_draw = training[
        "class_name"
    ].astype(
        str
    ).to_numpy()

    groups_draw = training[
        "sample_group"
    ].astype(
        str
    ).to_numpy()

    cv = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    for model_name, template in MODELS.items():

        pred_all = np.empty(
            len(
                training
            ),
            dtype=object,
        )

        pred_all[:] = None

        for train_idx, test_idx in cv.split(
            X_draw,
            y_draw,
            groups_draw,
        ):

            fitted = clone(
                template
            )

            fitted.fit(
                X_draw[
                    train_idx
                ],
                y_draw[
                    train_idx
                ],
            )

            pred_all[
                test_idx
            ] = fitted.predict(
                X_draw[
                    test_idx
                ]
            ).astype(
                str
            )

        all_scores = score_predictions(
            y_draw,
            pred_all,
        )

        veg_scores = score_vegetation_only(
            y_draw,
            pred_all,
        )

        draw_rows.append({
            "draw": draw,
            "seed": draw_seed,
            "model": model_name,
            "n_training": len(
                training
            ),
            "n_nonveg": int(
                np.sum(
                    y_draw
                    == NONVEG_CLASS
                )
            ),
            **all_scores,
            **veg_scores,
        })


draw_metrics = pd.DataFrame(
    draw_rows
)

draw_metrics.to_csv(
    OUTPUT_DIR
    / "ParentML_EMtbs_20draw_groupedOOF_metrics.csv",
    index=False,
)

summary = (
    draw_metrics.groupby(
        "model",
        as_index=False,
    )
    .agg(
        veg_accuracy_mean=(
            "veg_accuracy",
            "mean",
        ),
        veg_accuracy_sd=(
            "veg_accuracy",
            "std",
        ),
        veg_balanced_accuracy_mean=(
            "veg_balanced_accuracy",
            "mean",
        ),
        veg_macro_f1_mean=(
            "veg_macro_f1",
            "mean",
        ),
        veg_macro_f1_sd=(
            "veg_macro_f1",
            "std",
        ),
        veg_nonveg_leak_mean=(
            "veg_predicted_nonveg_fraction",
            "mean",
        ),
        all_macro_f1_mean=(
            "macro_f1",
            "mean",
        ),
    )
    .sort_values(
        "veg_macro_f1_mean",
        ascending=False,
    )
)

display(
    summary.round(
        4
    )
)


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:2524: UserWarning: y_pred contains classes not in

## 11. Distribution of vegetation macro F1 across balanced endmember draws


In [ ]:
# =============================================================================
# RESAMPLING-STABILITY FIGURE
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5,
    )
)

model_order = list(
    MODELS.keys()
)

positions = np.arange(
    len(
        model_order
    )
)

for i, model_name in enumerate(
    model_order
):

    values = (
        draw_metrics.loc[
            draw_metrics[
                "model"
            ].eq(
                model_name
            ),
            "veg_macro_f1",
        ]
        .to_numpy()
    )

    x = np.full(
        len(
            values
        ),
        i,
        dtype=float,
    )

    # deterministic horizontal jitter for legibility
    jitter = np.linspace(
        -0.12,
        0.12,
        len(
            values
        ),
    )

    ax.scatter(
        x + jitter,
        values,
        s=26,
        alpha=0.75,
    )

    ax.plot(
        [
            i - 0.18,
            i + 0.18,
        ],
        [
            np.mean(
                values
            ),
            np.mean(
                values
            ),
        ],
        linewidth=2.2,
    )

ax.set_xticks(
    positions
)

ax.set_xticklabels(
    model_order
)

ax.set_ylabel(
    "Vegetation-only macro F1"
)

ax.set_title(
    "Sensitivity to balanced NONVEGETATED subsampling",
    loc="left",
    fontsize=12,
    fontweight="bold",
    pad=20,
)

ax.text(
    0.0,
    1.015,
    "Twenty evenly stratified static-endmember draws; horizontal line = mean",
    transform=ax.transAxes,
    fontsize=9,
    ha="left",
    va="bottom",
)

ax.grid(
    axis="y",
    alpha=0.2,
)

ax.spines[
    "top"
].set_visible(
    False
)

ax.spines[
    "right"
].set_visible(
    False
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "ParentML_EMtbs_20draw_macroF1.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


## 12. Fixed mapping-draw grouped OOF predictions

The final raster model will use the deterministic seed-42 endmember draw. This cell generates its direct confusion matrices and classification reports.


In [ ]:
# =============================================================================
# FIXED-DRAW OOF
# =============================================================================

X_maptrain = mapping_training[
    IA_COLS
].to_numpy(
    dtype=np.float32
)

y_maptrain = mapping_training[
    "class_name"
].astype(
    str
).to_numpy()

groups_maptrain = mapping_training[
    "sample_group"
].astype(
    str
).to_numpy()

fixed_oof = {}

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

for model_name, template in MODELS.items():

    pred_all = np.empty(
        len(
            mapping_training
        ),
        dtype=object,
    )

    pred_all[:] = None

    for train_idx, test_idx in cv.split(
        X_maptrain,
        y_maptrain,
        groups_maptrain,
    ):

        fitted = clone(
            template
        )

        fitted.fit(
            X_maptrain[
                train_idx
            ],
            y_maptrain[
                train_idx
            ],
        )

        pred_all[
            test_idx
        ] = fitted.predict(
            X_maptrain[
                test_idx
            ]
        ).astype(
            str
        )

    fixed_oof[
        model_name
    ] = pred_all

    print(
        "\n",
        "=" * 90
    )

    print(
        model_name
    )

    print(
        "=" * 90
    )

    display(
        pd.DataFrame(
            classification_report(
                y_maptrain,
                pred_all,
                labels=ALL_CLASSES,
                output_dict=True,
                zero_division=0,
            )
        ).T.round(
            3
        )
    )


## 13. Fixed-draw confusion matrices

In [ ]:
# =============================================================================
# CONFUSION MATRICES
# =============================================================================

display_labels = [
    f"Parent {i}"
    for i in VEG_PARENT_CODES
] + [
    "NONVEGETATED",
]

for model_name in MODELS:

    pred_all = np.asarray(
        fixed_oof[
            model_name
        ]
    ).astype(
        str
    )

    cm = confusion_matrix(
        y_maptrain,
        pred_all,
        labels=ALL_CLASSES,
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            f"True {x}"
            for x in display_labels
        ],
        columns=[
            f"Pred {x}"
            for x in display_labels
        ],
    )

    print(
        "\n",
        model_name.upper()
    )

    display(
        cm_df
    )

    fig, ax = plt.subplots(
        figsize=(
            8.5,
            7.5,
        )
    )

    im = ax.imshow(
        cm,
        aspect="auto",
    )

    ax.set_xticks(
        np.arange(
            len(
                display_labels
            )
        )
    )

    ax.set_xticklabels(
        display_labels,
        rotation=40,
        ha="right",
    )

    ax.set_yticks(
        np.arange(
            len(
                display_labels
            )
        )
    )

    ax.set_yticklabels(
        display_labels
    )

    ax.set_xlabel(
        "Predicted class"
    )

    ax.set_ylabel(
        "True class"
    )

    ax.set_title(
        f"Grouped OOF confusion — {model_name}",
        loc="left",
        fontsize=12,
        fontweight="bold",
        pad=20,
    )

    ax.text(
        0.0,
        1.015,
        "Fixed balanced endmember draw; nine-class response space",
        transform=ax.transAxes,
        fontsize=9,
        ha="left",
        va="bottom",
    )

    for i in range(
        cm.shape[
            0
        ]
    ):
        for j in range(
            cm.shape[
                1
            ]
        ):
            ax.text(
                j,
                i,
                int(
                    cm[
                        i,
                        j
                    ]
                ),
                ha="center",
                va="center",
                fontsize=8,
            )

    fig.colorbar(
        im,
        ax=ax,
        fraction=0.035,
        pad=0.025,
    )

    fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR
        / f"ParentML_EMtbs_OOF_confusion_{model_name}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


## 14. Leave-one-vegetation-year-out on the fixed balanced draw

Because `NONVEGETATED` contains exactly the same three years as the vegetation support, holding out 2016/2018/2025 removes that year's observations from **both** sides of the classification problem.


In [ ]:
# =============================================================================
# FIXED-DRAW LEAVE-ONE-YEAR-OUT
# =============================================================================

loyo_rows = []

years_maptrain = mapping_training[
    "Year"
].to_numpy(
    dtype=int
)

for model_name, template in MODELS.items():

    for held_year in VEG_YEARS:

        train_mask = (
            years_maptrain
            != held_year
        )

        test_mask = (
            years_maptrain
            == held_year
        )

        fitted = clone(
            template
        )

        fitted.fit(
            X_maptrain[
                train_mask
            ],
            y_maptrain[
                train_mask
            ],
        )

        pred = fitted.predict(
            X_maptrain[
                test_mask
            ]
        ).astype(
            str
        )

        all_scores = score_predictions(
            y_maptrain[
                test_mask
            ],
            pred,
        )

        veg_scores = score_vegetation_only(
            y_maptrain[
                test_mask
            ],
            pred,
        )

        loyo_rows.append({
            "model": model_name,
            "held_out_year": held_year,
            "n_train": int(
                train_mask.sum()
            ),
            "n_test": int(
                test_mask.sum()
            ),
            **all_scores,
            **veg_scores,
        })


loyo_results = pd.DataFrame(
    loyo_rows
)

display(
    loyo_results.round(
        4
    )
)

loyo_results.to_csv(
    OUTPUT_DIR
    / "ParentML_EMtbs_leaveYearOut.csv",
    index=False,
)


## 15. Save the fixed balanced training set

In [ ]:
# =============================================================================
# SAVE FIXED DRAW
# =============================================================================

mapping_end_locations.to_csv(
    OUTPUT_DIR
    / "ParentML_EMtbs_selected_static_endmember_locations.csv",
    index=False,
)

mapping_training.to_csv(
    OUTPUT_DIR
    / "ParentML_EMtbs_fixed_training_table.csv",
    index=False,
)

class_lookup = pd.DataFrame({
    "class_code": list(
        range(
            1,
            10,
        )
    ),
    "class_name": ALL_CLASSES,
    "display_name": [
        f"Parent {i}"
        for i in VEG_PARENT_CODES
    ] + [
        "NONVEGETATED",
    ],
})

class_lookup.to_csv(
    OUTPUT_DIR
    / "ParentML_EMtbs_class_lookup.csv",
    index=False,
)

display(
    class_lookup
)


## 16. Fit the final random forest

In [ ]:
# =============================================================================
# FINAL MODEL
# =============================================================================

mapping_model = clone(
    MODELS[
        MODEL_FOR_MAPPING
    ]
)

t0 = time.perf_counter()

mapping_model.fit(
    X_maptrain,
    y_maptrain,
)

print(
    "Final mapping model:",
    MODEL_FOR_MAPPING
)

print(
    "Training rows:",
    len(
        mapping_training
    )
)

print(
    "Fit time:",
    f"{(time.perf_counter() - t0) / 60:.2f} min"
)


## 17. Raster helpers and blockwise classification

In [ ]:
# =============================================================================
# BLOCKWISE CLASSIFICATION
# =============================================================================

def grids_match(
    src_a,
    src_b,
):
    return (
        src_a.width == src_b.width
        and src_a.height == src_b.height
        and src_a.transform == src_b.transform
        and src_a.crs == src_b.crs
    )


def predictive_entropy(
    proba,
):
    eps = 1e-12

    return -np.sum(
        proba
        * np.log(
            np.maximum(
                proba,
                eps,
            )
        ),
        axis=1,
    )


def classify_year(
    year,
    model,
):

    src_path = annual_k2_path(
        year
    )

    class_path = (
        OUTPUT_DIR
        / f"ParentML_EMtbs_RF_{year}_class.tif"
    )

    uncertainty_path = (
        OUTPUT_DIR
        / f"ParentML_EMtbs_RF_{year}_uncertainty.tif"
    )

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"CLASSIFYING {year}"
    )

    print(
        "=" * 100
    )

    year_start = time.perf_counter()

    with rasterio.open(
        src_path
    ) as src, rasterio.open(
        KEEP_MASK
    ) as mask_src:

        indexes = ia_band_indexes(
            src,
            IA_COLS,
        )

        use_direct_mask = grids_match(
            src,
            mask_src,
        )

        mask_vrt = None

        if not use_direct_mask:

            mask_vrt = WarpedVRT(
                mask_src,
                crs=src.crs,
                transform=src.transform,
                width=src.width,
                height=src.height,
                resampling=Resampling.nearest,
            )

        blocks = list(
            src.block_windows(
                1
            )
        )

        n_blocks = len(
            blocks
        )

        class_profile = src.profile.copy()

        class_profile.update(
            driver="GTiff",
            count=1,
            dtype="uint8",
            nodata=0,
            compress=COMPRESS,
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        uncertainty_profile = src.profile.copy()

        uncertainty_profile.update(
            driver="GTiff",
            count=3,
            dtype="float32",
            nodata=-9999.0,
            compress=COMPRESS,
            predictor=3,
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        model_classes = np.asarray(
            model.classes_,
            dtype=object,
        )

        with rasterio.open(
            class_path,
            "w",
            **class_profile,
        ) as class_dst, rasterio.open(
            uncertainty_path,
            "w",
            **uncertainty_profile,
        ) as uncertainty_dst:

            class_dst.set_band_description(
                1,
                "predicted_class_code",
            )

            uncertainty_dst.set_band_description(
                1,
                "max_probability",
            )

            uncertainty_dst.set_band_description(
                2,
                "probability_margin",
            )

            uncertainty_dst.set_band_description(
                3,
                "prediction_entropy",
            )

            total_keep = 0
            total_valid = 0

            for block_number, (
                _,
                window,
            ) in enumerate(
                blocks,
                start=1,
            ):

                if use_direct_mask:
                    keep = mask_src.read(
                        1,
                        window=window,
                        masked=False,
                    )
                else:
                    keep = mask_vrt.read(
                        1,
                        window=window,
                        masked=False,
                    )

                keep_valid = (
                    np.isfinite(
                        keep
                    )
                    & (
                        keep > 0
                    )
                )

                total_keep += int(
                    keep_valid.sum()
                )

                features = src.read(
                    indexes,
                    window=window,
                    masked=False,
                ).astype(
                    np.float32,
                    copy=False,
                )

                features = np.moveaxis(
                    features,
                    0,
                    -1,
                )

                valid = (
                    keep_valid
                    & np.isfinite(
                        features
                    ).all(
                        axis=2
                    )
                )

                if src.nodata is not None:
                    valid &= ~np.any(
                        np.isclose(
                            features,
                            src.nodata,
                        ),
                        axis=2,
                    )

                n_valid = int(
                    valid.sum()
                )

                total_valid += n_valid

                class_block = np.zeros(
                    (
                        int(
                            window.height
                        ),
                        int(
                            window.width
                        ),
                    ),
                    dtype=np.uint8,
                )

                uncertainty_block = np.full(
                    (
                        3,
                        int(
                            window.height
                        ),
                        int(
                            window.width
                        ),
                    ),
                    -9999.0,
                    dtype=np.float32,
                )

                if n_valid > 0:

                    X_block = features[
                        valid
                    ]

                    proba = model.predict_proba(
                        X_block
                    ).astype(
                        np.float32,
                        copy=False,
                    )

                    winner_idx = np.argmax(
                        proba,
                        axis=1,
                    )

                    winner_class = model_classes[
                        winner_idx
                    ]

                    winner_code = np.fromiter(
                        (
                            CLASS_TO_CODE[
                                str(
                                    cls
                                )
                            ]
                            for cls in winner_class
                        ),
                        dtype=np.uint8,
                        count=len(
                            winner_class
                        ),
                    )

                    class_block[
                        valid
                    ] = winner_code

                    sorted_p = np.sort(
                        proba,
                        axis=1,
                    )

                    max_p = sorted_p[
                        :,
                        -1
                    ]

                    margin = (
                        sorted_p[
                            :,
                            -1
                        ]
                        - sorted_p[
                            :,
                            -2
                        ]
                    )

                    entropy = predictive_entropy(
                        proba
                    )

                    uncertainty_block[
                        0
                    ][
                        valid
                    ] = max_p

                    uncertainty_block[
                        1
                    ][
                        valid
                    ] = margin

                    uncertainty_block[
                        2
                    ][
                        valid
                    ] = entropy.astype(
                        np.float32
                    )

                class_dst.write(
                    class_block,
                    1,
                    window=window,
                )

                uncertainty_dst.write(
                    uncertainty_block,
                    window=window,
                )

                if (
                    block_number == 1
                    or block_number % 25 == 0
                    or block_number == n_blocks
                ):

                    elapsed = (
                        time.perf_counter()
                        - year_start
                    )

                    rate = (
                        block_number
                        / elapsed
                        if elapsed > 0
                        else np.nan
                    )

                    eta = (
                        (
                            n_blocks
                            - block_number
                        )
                        / rate
                        if rate > 0
                        else np.nan
                    )

                    print(
                        f"{year} | "
                        f"{block_number:,}/{n_blocks:,} "
                        f"({100 * block_number / n_blocks:5.1f}%) | "
                        f"elapsed {elapsed / 60:6.1f} min | "
                        f"ETA {eta / 60:6.1f} min | "
                        f"classified {total_valid:,} px"
                    )

        if mask_vrt is not None:
            mask_vrt.close()

    runtime = (
        time.perf_counter()
        - year_start
    ) / 60.0

    print(
        f"{year} complete in {runtime:.1f} min"
    )

    return {
        "Year": year,
        "runtime_minutes": runtime,
        "keep_pixels": total_keep,
        "classified_pixels": total_valid,
        "class_path": str(
            class_path
        ),
        "uncertainty_path": str(
            uncertainty_path
        ),
    }


## 18. Classify all annual rasters, 2016–2025

In [ ]:
# =============================================================================
# MAP ALL YEARS
# =============================================================================

map_results = []

all_start = time.perf_counter()

for year in YEARS_TO_MAP:

    result = classify_year(
        year,
        mapping_model,
    )

    map_results.append(
        result
    )

    pd.DataFrame(
        map_results
    ).to_csv(
        OUTPUT_DIR
        / "ParentML_EMtbs_mapping_progress.csv",
        index=False,
    )


map_results = pd.DataFrame(
    map_results
)

display(
    map_results
)

map_results.to_csv(
    OUTPUT_DIR
    / "ParentML_EMtbs_allYears_outputs.csv",
    index=False,
)

print(
    "Total mapping runtime:",
    f"{(time.perf_counter() - all_start) / 60:.1f} min"
)


## 19. Annual class-frequency summaries

In [ ]:
# =============================================================================
# CLASS FREQUENCIES
# =============================================================================

frequency_rows = []

for year in YEARS_TO_MAP:

    path = (
        OUTPUT_DIR
        / f"ParentML_EMtbs_RF_{year}_class.tif"
    )

    with rasterio.open(
        path
    ) as src:

        arr = src.read(
            1,
            masked=False,
        )

    valid = (
        arr > 0
    )

    n_valid = int(
        valid.sum()
    )

    codes, counts = np.unique(
        arr[
            valid
        ],
        return_counts=True,
    )

    summary_year = pd.DataFrame({
        "Year": year,
        "class_code": codes.astype(
            int
        ),
        "n_pixels": counts.astype(
            np.int64
        ),
    })

    summary_year[
        "class_name"
    ] = summary_year[
        "class_code"
    ].map(
        CODE_TO_CLASS
    )

    summary_year[
        "percent"
    ] = (
        100.0
        * summary_year[
            "n_pixels"
        ]
        / n_valid
    )

    frequency_rows.append(
        summary_year
    )

    print(
        "\n",
        year
    )

    display(
        summary_year
    )


frequency_df = pd.concat(
    frequency_rows,
    ignore_index=True,
)

frequency_df.to_csv(
    OUTPUT_DIR
    / "ParentML_EMtbs_allYears_class_frequencies.csv",
    index=False,
)


## 20. Annual uncertainty summaries

In [ ]:
# =============================================================================
# UNCERTAINTY SUMMARIES
# =============================================================================

uncertainty_rows = []

band_names = {
    1: "max_probability",
    2: "probability_margin",
    3: "prediction_entropy",
}

for year in YEARS_TO_MAP:

    path = (
        OUTPUT_DIR
        / f"ParentML_EMtbs_RF_{year}_uncertainty.tif"
    )

    with rasterio.open(
        path
    ) as src:

        for band_no, metric in band_names.items():

            arr = src.read(
                band_no,
                masked=True,
            )

            values = arr.compressed()

            uncertainty_rows.append({
                "Year": year,
                "metric": metric,
                "n": len(
                    values
                ),
                "mean": float(
                    np.mean(
                        values
                    )
                ),
                "sd": float(
                    np.std(
                        values
                    )
                ),
                "p05": float(
                    np.quantile(
                        values,
                        0.05,
                    )
                ),
                "p25": float(
                    np.quantile(
                        values,
                        0.25,
                    )
                ),
                "median": float(
                    np.median(
                        values
                    )
                ),
                "p75": float(
                    np.quantile(
                        values,
                        0.75,
                    )
                ),
                "p95": float(
                    np.quantile(
                        values,
                        0.95,
                    )
                ),
            })


uncertainty_summary = pd.DataFrame(
    uncertainty_rows
)

display(
    uncertainty_summary.round(
        4
    )
)

uncertainty_summary.to_csv(
    OUTPUT_DIR
    / "ParentML_EMtbs_allYears_uncertainty_summary.csv",
    index=False,
)

print(
    "Nine-class maximum entropy:",
    np.log(
        len(
            ALL_CLASSES
        )
    ),
)


## 21. Full-resolution classified maps

In [ ]:
# =============================================================================
# FULL-RESOLUTION MAP FIGURES
# =============================================================================

from matplotlib.colors import ListedColormap, BoundaryNorm

display_names = [
    f"Parent {i}"
    for i in VEG_PARENT_CODES
] + [
    "NONVEGETATED",
]

base_cmap = plt.get_cmap(
    "tab10",
    len(
        ALL_CLASSES
    ),
)

colors = [
    base_cmap(
        i
    )
    for i in range(
        len(
            ALL_CLASSES
        )
    )
]

class_cmap = ListedColormap(
    colors
)

class_cmap.set_bad(
    alpha=0
)

class_norm = BoundaryNorm(
    np.arange(
        0.5,
        len(
            ALL_CLASSES
        ) + 1.5,
        1.0,
    ),
    len(
        ALL_CLASSES
    ),
)

for year in YEARS_TO_MAP:

    class_path = (
        OUTPUT_DIR
        / f"ParentML_EMtbs_RF_{year}_class.tif"
    )

    with rasterio.open(
        class_path
    ) as src:

        arr = src.read(
            1,
            masked=False,
        )

        arr = np.ma.masked_where(
            arr == 0,
            arr,
        )

        extent = [
            src.bounds.left,
            src.bounds.right,
            src.bounds.bottom,
            src.bounds.top,
        ]

    fig, ax = plt.subplots(
        figsize=(
            16,
            16,
        )
    )

    image = ax.imshow(
        arr,
        cmap=class_cmap,
        norm=class_norm,
        extent=extent,
        origin="upper",
        interpolation="none",
        rasterized=True,
    )

    ax.set_title(
        f"{year} predicted vegetation + nonvegetated classes",
        loc="left",
        fontsize=15,
        fontweight="bold",
        pad=22,
    )

    ax.text(
        0.0,
        1.01,
        (
            "Random forest from IA45; mask v2; "
            "balanced static endmember support; native 10 m classification"
        ),
        transform=ax.transAxes,
        fontsize=10,
        ha="left",
        va="bottom",
    )

    ax.set_xlabel(
        "Easting (m)"
    )

    ax.set_ylabel(
        "Northing (m)"
    )

    ax.ticklabel_format(
        style="plain",
        axis="both",
        useOffset=False,
    )

    cbar = fig.colorbar(
        image,
        ax=ax,
        fraction=0.03,
        pad=0.025,
        ticks=np.arange(
            1,
            len(
                ALL_CLASSES
            ) + 1,
        ),
    )

    cbar.set_label(
        "Predicted class"
    )

    cbar.ax.set_yticklabels(
        display_names
    )

    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )

    fig.subplots_adjust(
        left=0.08,
        right=0.86,
        bottom=0.07,
        top=0.91,
    )

    path = (
        OUTPUT_DIR
        / f"ParentML_EMtbs_RF_{year}_map_fullres.png"
    )

    fig.savefig(
        path,
        dpi=400,
        bbox_inches="tight",
    )

    plt.show()

    plt.close(
        fig
    )

    del arr

    print(
        "Wrote:",
        path
    )


## Interpretation

This `_EMtbs` experiment isolates the support question much more cleanly than the previous many-endmember-class formulation.

The principal comparisons are:

- original vegetation-only baseline;
- mask-v2 vegetation-only performance;
- `_EMtbs` nine-class performance.

The key outcomes are:

1. whether balanced `NONVEGETATED` support preserves or improves vegetation macro F1;
2. whether true vegetation is still disproportionately routed to nonvegetated;
3. whether leave-year-out behavior improves once both vegetation and nonvegetated training occupy exactly the same three annual domains;
4. whether the mapped nonvegetated class removes obvious bare/dark surfaces without becoming a landscape-wide sink;
5. whether uncertainty becomes better localized.

Because the final mapping draw is fixed in advance (`seed=42`) and the CV experiment evaluates 20 independent balanced draws, improvements cannot be attributed to cherry-picking a favorable endmember subset.
